# Battery Discharge Analysis


```SQL
-- STEP 0: Deduplicate raw bloq records to keep the most recent row per bloq per hour
WITH ranked AS
  (SELECT bloqs_id,
          partner,
          active,
          LOCATION,
          metadata,
          details,
          firmwaredetails,
          date_trunc('hour', system_updated_at) AS hour_bucket,
          ROW_NUMBER() OVER (PARTITION BY bloqs_id, date_trunc('hour', system_updated_at)
                             ORDER BY system_updated_at DESC) AS rn
   FROM prod_bo2dl_bloqs_gluedb_prepared.prod_bo2dl_bloqs_prepared_iceberg_t
   WHERE 1 = 1
     AND system_updated_at >= current_date - INTERVAL '1' MONTH
     AND powersource = 'BATTERY'
     AND lifecyclestatus IS NOT NULL
     AND lifecyclestatus <> 'terminated'
     AND active = 'true'),
-- STEP 1: Keep only the latest hourly row and ensure both batteries exist
pre_filtered AS
  (SELECT *
   FROM ranked
   WHERE rn = 1
     AND lower(firmwaredetails) LIKE '%bat1%'
     AND lower(firmwaredetails) LIKE '%bat2%'),
-- STEP 2: Flatten battery JSON array into one row per battery per hour
daily_battery AS
  (SELECT pf.hour_bucket,
          pf.bloqs_id,
 CASE
     WHEN json_extract_scalar(pf.metadata, '$.bloqitName') IS NULL THEN NULL
     WHEN length(json_extract_scalar(pf.metadata, '$.bloqitName')) > 5 THEN substr(json_extract_scalar(pf.metadata, '$.bloqitName'), 1, length(json_extract_scalar(pf.metadata, '$.bloqitName')) - 5)
     ELSE json_extract_scalar(pf.metadata, '$.bloqitName')
 END AS partner,
 json_extract_scalar(pf.metadata, '$.bloqitName') AS bloq_name,
 json_extract_scalar(pf.firmwaredetails, '$.firmwareVersion') AS fw_version,
 CASE
    WHEN json_extract_scalar(pf.firmwaredetails, '$.firmwareVersion') LIKE '7.1.%'
         AND json_extract_scalar(pf.firmwaredetails, '$.firmwareVersion') NOT IN ('7.1.1', '7.1.2')
    THEN 'recent'
    ELSE 'old'
END AS fw_version_type,
 json_extract_scalar(bat, '$.batteryName') AS battery_name,
 CAST(json_extract_scalar(bat, '$.capacityStateOffCharge') AS DOUBLE) AS soc,
 json_extract_scalar(bat, '$.voltage') AS voltage,
 json_extract_scalar(bat, '$.temperature') AS temperature,
 json_extract_scalar(bat, '$.extraInfo.battery_serial_number') AS battery_serial_number,
 json_extract_scalar(pf.details, '$.country') AS country,
 json_extract_scalar(pf.details, '$.address') AS address,
 coalesce(json_extract_scalar(pf.metadata, '$.apm_name'), json_extract_scalar(pf.metadata, '$.vintedCode')) AS point_code
   FROM pre_filtered pf
   CROSS JOIN UNNEST(CAST(json_parse(json_extract_scalar(pf.firmwaredetails, '$.battery')) AS array(JSON))) AS t(bat)
   WHERE json_extract_scalar(bat, '$.batteryName') IS NOT NULL
     AND json_extract_scalar(bat, '$.batteryName') <> ''
     AND json_extract_scalar(bat, '$.capacityStateOffCharge') IS NOT NULL
     AND json_extract_scalar(bat, '$.capacityStateOffCharge') <> ''),
-- STEP 3: Deduplicate events and aggregate rents per bloq per hour
events_ranked AS
  (SELECT rent,
          bloq,
          CAST(date_trunc('hour', from_iso8601_timestamp(timestamp)) AS timestamp) AS timestp,
          ROW_NUMBER() OVER (PARTITION BY events_id
                             ORDER BY pipeline_ingested_at DESC) AS rn
   FROM prod_bo2dl_events_gluedb_prepared.prod_bo2dl_events_prepared_iceberg_t
   WHERE bloq IS NOT NULL),
rents_table AS
  (SELECT timestp,
          bloq,
          count(DISTINCT rent) AS rents
   FROM events_ranked
   WHERE rn = 1
   GROUP BY bloq,
            timestp)
SELECT d.*,
       r.rents
FROM daily_battery d
LEFT JOIN rents_table r ON r.bloq = d.bloqs_id
AND r.timestp = d.hour_bucket;
```

In [1]:
# ============================================================
# Imports
# ============================================================~

#!pip install gspread gspread-dataframe

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import gspread


from google.colab import auth, drive
from google.auth import default
from gspread_dataframe import get_as_dataframe
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    balanced_accuracy_score,
    accuracy_score,
)

# =========================
# PERSISTENT MODEL PATH (Google Drive)
# =========================
from google.colab import drive
drive.mount("/content/drive")

MODEL_PATH = "/content/drive/MyDrive/battery_models/battery_risk_model.joblib"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

# ============================================================
# Authenticate Google
# ============================================================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

Mounted at /content/drive


In [2]:
# ============================================================
# Load TRAIN dataset
# ============================================================
TRAIN_DATA_PATH = "/content/drive/MyDrive/battery_models/dataset/202602181520_battery_data_train.csv"

try:
    df = pd.read_csv(TRAIN_DATA_PATH)
    print(f"Successfully loaded {TRAIN_DATA_PATH}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"File not found: {TRAIN_DATA_PATH}. "
        "Please check the path and try again."
    )
except Exception as e:
    raise RuntimeError(f"Error loading training CSV: {e}")

# Standardise column names
df = df.rename(
    columns={
        "hour_bucket": "timestamp"
    }
)

# ============================================================
# Load NEW dataset (for scoring)
# ============================================================
NEW_DATASET_PATH = "/content/drive/MyDrive/battery_models/dataset/202604210854_battery_deploy.csv"  # update this!

df_new_hourly = pd.read_csv(
    NEW_DATASET_PATH,
    engine="python",
    on_bad_lines="skip"
)

df_new_hourly["hour_bucket"] = pd.to_datetime(
    df_new_hourly["hour_bucket"],
    errors="coerce"
)

# Standardise column names
df_new_hourly = df_new_hourly.rename(
    columns={
        "hour_bucket": "timestamp"
    }
)

# Schema validation
REQUIRED_COLS = [
    "timestamp",
    "bloqs_id",
    "battery_name",
    "soc",
    "voltage",
    "temperature",
    "rents"
]

missing_cols = [c for c in REQUIRED_COLS if c not in df_new_hourly.columns]
if missing_cols:
    raise ValueError(
        f"Missing required columns in NEW dataset: {missing_cols}"
    )

print(f"Successfully loaded {NEW_DATASET_PATH}")


Successfully loaded /content/drive/MyDrive/battery_models/dataset/202602181520_battery_data_train.csv
Successfully loaded /content/drive/MyDrive/battery_models/dataset/202604210854_battery_deploy.csv


### Data Cleaning

In [3]:
# ===========================
# CLEAN BOTH DATASETS
# ===========================

def clean_battery_dataset(df: pd.DataFrame, name: str = "dataset") -> pd.DataFrame:
    df0 = df.copy()

    # Timestamp
    df0["timestamp"] = pd.to_datetime(df0["timestamp"], errors="coerce")

    # Sort
    sort_cols = ["bloqs_id", "timestamp"]
    if "battery_name" in df0.columns:
        sort_cols.insert(1, "battery_name")

    df0 = df0.sort_values(by=sort_cols).reset_index(drop=True)

    # ---- Battery serial cleaning ----
    # do NOT drop rows when serial is missing; just normalize invalid strings to NaN
    if "battery_serial_number" in df0.columns:
        serial = df0["battery_serial_number"].astype(str).str.strip()

        invalid = serial.isin(["", "N/A", "nan", "None"])
        serial = serial.mask(invalid, np.nan)

        df0["battery_serial_number"] = serial

    # Remove invalid timestamp rows
    df0 = df0[df0["timestamp"].notna()].copy()

    # Remove invalid soc / voltage
    df0 = df0[
        (df0["soc"].notna()) &
        (df0["soc"] >= 0) &
        (df0["voltage"].notna()) &
        (df0["voltage"] > 0)
    ].copy()

    # Print summary
    orig_rows = len(df)
    orig_bloqs = df["bloqs_id"].nunique() if "bloqs_id" in df.columns else 0
    cleaned_rows = len(df0)
    cleaned_bloqs = df0["bloqs_id"].nunique() if "bloqs_id" in df0.columns else 0

    print(f"\n[{name}] Original size: {orig_rows} rows | bloqs: {orig_bloqs}")
    print(f"[{name}] Cleaned  size: {cleaned_rows} rows | bloqs: {cleaned_bloqs}")

    return df0

# ---- Apply to both datasets ----
df_clean = clean_battery_dataset(df, name="TRAIN (df)")
df_new_clean = clean_battery_dataset(df_new_hourly, name="NEW (df_new_hourly)")





[TRAIN (df)] Original size: 7493118 rows | bloqs: 2172
[TRAIN (df)] Cleaned  size: 6653764 rows | bloqs: 2152

[NEW (df_new_hourly)] Original size: 4900543 rows | bloqs: 2907
[NEW (df_new_hourly)] Cleaned  size: 4854084 rows | bloqs: 2904


### Identify Battery Cycles

In [4]:
# ============================================================
# SESSIONIZATION (serial-number based) + days_left
# FIX: use last known Serial Number when current is NaN (forward fill)
# ============================================================

def add_sessions_and_days_left(
    df_in: pd.DataFrame,
    name: str = "dataset",
    censor_tol: str = "1h"
) -> pd.DataFrame:
    df = df_in.copy()

    # ---- Required columns ----
    if "bloqs_id" not in df.columns:
        raise KeyError(f"[{name}] Missing required column: bloqs_id")
    if "timestamp" not in df.columns:
        raise KeyError(f"[{name}] Missing required column: timestamp")

    # Ensure timestamp is datetime and sorted
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).copy()

    # Determine grouping keys for "battery identity"
    group_keys = ["bloqs_id"]
    if "battery_name" in df.columns:
        group_keys.append("battery_name")

    sort_cols = group_keys + ["timestamp"]
    df = df.sort_values(by=sort_cols).reset_index(drop=True)

    # Serial number handling (only if present)
    has_serial = "battery_serial_number" in df.columns
    if has_serial:
        df["battery_serial_number"] = (
            df["battery_serial_number"]
            .astype(str)
            .str.strip()
        )

        invalid_serials = {"", "N/A", "nan", "None"}
        df.loc[df["battery_serial_number"].isin(invalid_serials), "battery_serial_number"] = np.nan

        # Carry forward last known serial (per bloq+battery)
        df["battery_serial_number_filled"] = (
            df.groupby(group_keys, observed=True)["battery_serial_number"].ffill()
        )

        # Detect new sessions by serial change
        df["prev_serial"] = (
            df.groupby(group_keys, observed=True)["battery_serial_number_filled"].shift(1)
        )

        df["is_new_session"] = (
            df["battery_serial_number_filled"].notna() &
            df["prev_serial"].notna() &
            (df["battery_serial_number_filled"] != df["prev_serial"])
        ).astype(int)

        # First row per battery identity group is always session 0
        first_idx = df.groupby(group_keys, observed=True).head(1).index
        df.loc[first_idx, "is_new_session"] = 0

        # Assign session IDs
        df["session_id"] = df.groupby(group_keys, observed=True)["is_new_session"].cumsum()

        # Use filled serial downstream (so Feb rows keep Jan SN)
        df["battery_serial_number"] = df["battery_serial_number_filled"]
        df = df.drop(columns=["battery_serial_number_filled"])

    else:
        df["battery_serial_number"] = np.nan
        df["prev_serial"] = np.nan
        df["is_new_session"] = 0
        df["session_id"] = 0

    # Compute session start and end times using transform
    grp = group_keys + ["session_id"]
    df["battery_start_time"] = df.groupby(grp, observed=True)["timestamp"].transform("min")
    df["battery_end_time_raw"] = df.groupby(grp, observed=True)["timestamp"].transform("max")

    # ---------------------------
    # CENSORING FIX
    # ---------------------------
    global_end = df["timestamp"].max()
    CENSOR_TOL = pd.Timedelta(censor_tol)

    df["is_censored"] = (df["battery_end_time_raw"] >= (global_end - CENSOR_TOL)).astype(int)

    df["battery_end_time"] = df["battery_end_time_raw"]
    df.loc[df["is_censored"] == 1, "battery_end_time"] = pd.NaT

    # Compute days_left ONLY for non-censored sessions
    df["days_left"] = np.nan
    mask = df["battery_end_time"].notna()
    df.loc[mask, "days_left"] = (
        (df.loc[mask, "battery_end_time"] - df.loc[mask, "timestamp"])
        .dt.total_seconds() / (24 * 3600)
    ).round(1)

    # Ensure expected + optional columns exist (schema-safe)
    required_and_optional_cols = [
        "soc",
        "voltage",
        "battery_name",
        "country",
        "address",
        "point_code",
    ]

    for col in required_and_optional_cols:
        if col not in df.columns:
            df[col] = np.nan


    out_cols = [
        "timestamp",
        "partner",
        "bloq_name",
        "fw_version_type",
        "bloqs_id",
        "battery_name",
        "battery_serial_number",
        "soc",
        "voltage",
        "temperature",
        "rents",
        "days_left",
        "battery_start_time",
        "battery_end_time",
        "session_id",
        "is_censored",
        "country",
        "address",
        "point_code"
    ]

    df_final_out = df[out_cols].copy()

    print(f"\n[{name}] Battery sessions computed (censoring handled).")
    print(f"[{name}] Remaining bloqs: {df_final_out['bloqs_id'].nunique()}")
    print(f"[{name}] Censored sessions: {df_final_out.groupby(grp, observed=True)['is_censored'].max().sum()}")

    return df_final_out

# ============================================================
# APPLY TO BOTH DATASETS
# ============================================================

# TRAIN dataset
df_final = add_sessions_and_days_left(df_clean, name="TRAIN (df_clean)", censor_tol="1h")

# NEW dataset
df_new_final = add_sessions_and_days_left(df_new_clean, name="NEW (df_new_clean)", censor_tol="1h")



[TRAIN (df_clean)] Battery sessions computed (censoring handled).
[TRAIN (df_clean)] Remaining bloqs: 2152
[TRAIN (df_clean)] Censored sessions: 3825

[NEW (df_new_clean)] Battery sessions computed (censoring handled).
[NEW (df_new_clean)] Remaining bloqs: 2904
[NEW (df_new_clean)] Censored sessions: 6885


### Battery Risk Model

In [5]:
# =========================
# MODE SWITCH
# =========================
MODE = "score"   # "train" or "score"




In [6]:
# ============================================================
# HYBRID MODEL (Collinearity-fixed: drop volt_min/volt_max/volt_std)
# ============================================================

# CONFIG
HORIZON = 7
SOC_THRESHOLD = 60
RANDOM_STATE = 42
ALERT_THRESHOLD = 0.8

NUM_FEATURES = [
    "volt_mean",
    "volt_slope_3d",
    "volt_slope_7d",
    "volt_vol_7d",
    "soc_slope_3d",
    "soc_slope_7d",
    "soc_vol_7d",
    "temp_mean",
    "rents_sum",
]
CAT_FEATURES = ["fw_version_type"]
FEATURES_ALL = NUM_FEATURES + CAT_FEATURES

today_str = datetime.today().strftime("%Y%m%d")
OUTPUT_CSV = f"battery_risk_{today_str}.csv"


# =========================
# HELPERS
# =========================
def slope(y: pd.Series) -> float:
    if len(y) < 2:
        return 0.0
    x = np.arange(len(y))
    return float(np.polyfit(x, y, 1)[0])


def build_daily_features(df_hourly: pd.DataFrame) -> pd.DataFrame:
    required = [
        "timestamp",
        "bloqs_id",
        "battery_name",
        "session_id",
        "soc",
        "voltage",
        "temperature",
        "rents",
        "fw_version_type",
    ]
    missing = [c for c in required if c not in df_hourly.columns]
    if missing:
        raise ValueError(f"Missing columns for feature build: {missing}")

    optional = ["country", "address", "point_code"]
    cols_to_take = required + [c for c in optional if c in df_hourly.columns]
    df = df_hourly[cols_to_take].copy()

    for c in optional:
        if c not in df.columns:
            df[c] = np.nan

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).copy()
    df["date"] = df["timestamp"].dt.floor("D")

    df["session_id"] = pd.to_numeric(df["session_id"], errors="coerce").fillna(0).astype(int)

    df["soc"] = pd.to_numeric(df["soc"], errors="coerce")
    df["voltage"] = pd.to_numeric(df["voltage"], errors="coerce")
    df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")  # Celsius
    df["rents"] = pd.to_numeric(df["rents"], errors="coerce")

    df["fw_version_type"] = df["fw_version_type"].astype("string")

    daily = (
        df.groupby(["bloqs_id", "battery_name", "session_id", "date"], observed=True, sort=False)
        .agg(
            country=("country", "first"),
            address=("address", "first"),
            point_code=("point_code", "first"),
            fw_version_type=("fw_version_type", "first"),
            soc_mean=("soc", "mean"),
            soc_min=("soc", "min"),
            volt_mean=("voltage", "mean"),
            temp_mean=("temperature", "mean"),
            rents_sum=("rents", "sum"),
        )
        .reset_index()
    )

    daily = daily.sort_values(["bloqs_id", "battery_name", "session_id", "date"]).reset_index(drop=True)
    g = daily.groupby(["bloqs_id", "battery_name", "session_id"], observed=True, sort=False)

    daily["volt_slope_3d"] = g["volt_mean"].transform(
        lambda s: s.rolling(3, min_periods=2).apply(slope, raw=False)
    )
    daily["volt_slope_7d"] = g["volt_mean"].transform(
        lambda s: s.rolling(7, min_periods=3).apply(slope, raw=False)
    )
    daily["volt_vol_7d"] = g["volt_mean"].transform(
        lambda s: s.rolling(7, min_periods=3).std()
    ).fillna(0)

    daily["soc_slope_3d"] = g["soc_mean"].transform(
        lambda s: s.rolling(3, min_periods=2).apply(slope, raw=False)
    )
    daily["soc_slope_7d"] = g["soc_mean"].transform(
        lambda s: s.rolling(7, min_periods=3).apply(slope, raw=False)
    )
    daily["soc_vol_7d"] = g["soc_mean"].transform(
        lambda s: s.rolling(7, min_periods=3).std()
    ).fillna(0)

    return daily


def add_soc_threshold_label(daily: pd.DataFrame) -> pd.DataFrame:
    out = daily.copy()
    g = out.groupby(["bloqs_id", "battery_name", "session_id"], observed=True, sort=False)
    out["future_min_soc"] = g["soc_min"].transform(
        lambda s: s.shift(-1).rolling(HORIZON, min_periods=1).min()
    )
    out["y_risk"] = (out["future_min_soc"] < SOC_THRESHOLD).astype(int)
    return out


def session_split(df_model: pd.DataFrame):
    sessions = (
        df_model[["bloqs_id", "battery_name", "session_id"]]
        .drop_duplicates()
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    cut = int(0.8 * len(sessions))
    train_sessions = sessions.iloc[:cut]
    test_sessions = sessions.iloc[cut:]

    train = df_model.merge(train_sessions, on=["bloqs_id", "battery_name", "session_id"], how="inner")
    test = df_model.merge(test_sessions, on=["bloqs_id", "battery_name", "session_id"], how="inner")
    return train, test


# =========================
# PRINT REPORTING
# =========================
def get_feature_names(pipe: Pipeline):
    prep = pipe.named_steps["prep"]
    try:
        return prep.get_feature_names_out()
    except Exception:
        return None


def print_variable_importance(pipe: Pipeline, top_n: int = 30):
    clf = pipe.named_steps["clf"]
    if not hasattr(clf, "coef_"):
        print("\n[Importance] No coefficients available for this model.")
        return

    names = get_feature_names(pipe)
    coefs = clf.coef_.ravel()

    print("\n[Importance] Top coefficients by |coef| (numerics are standardized):")
    if names is not None and len(names) == len(coefs):
        s = pd.Series(coefs, index=names).sort_values(key=np.abs, ascending=False)
        print(s.head(top_n).to_string())
    else:
        idx = np.argsort(np.abs(coefs))[::-1][:top_n]
        for i in idx:
            print(f"coef[{i}] = {coefs[i]: .5f}")


def print_model_score_or_summary(pipe: Pipeline, df_for_scoring: pd.DataFrame, title: str = ""):
    if title:
        print("\n" + title)
        print("-" * len(title))

    # If labeled, print real model score + threshold metrics
    if "y_risk" in df_for_scoring.columns:
        # Drop rows with missing features
        eval_df = df_for_scoring.dropna(subset=FEATURES_ALL + ["y_risk"]).copy()
        if len(eval_df) == 0 or eval_df["y_risk"].nunique() < 2:
            print("Not enough labeled data to compute metrics.")
            return

        y = eval_df["y_risk"].astype(int)
        X = eval_df[FEATURES_ALL]

        # Predicted probabilities
        p = pipe.predict_proba(X)[:, 1]

        # Ranking metrics (threshold independent)
        print("Rows:", len(eval_df))
        print("Positive rate:", round(float(y.mean()), 4))
        print("ROC AUC:", round(float(roc_auc_score(y, p)), 3))
        print("PR  AUC:", round(float(average_precision_score(y, p)), 3))

        # Threshold-based metrics (using fixed ALERT_THRESHOLD)
        yhat = (p >= ALERT_THRESHOLD).astype(int)

        print(f"\n--- Threshold Metrics (ALERT_THRESHOLD = {ALERT_THRESHOLD:.2f}) ---")
        print("Alert rate:", round(float(yhat.mean()), 3))
        print("Recall:", round(float(recall_score(y, yhat, zero_division=0)), 3))
        print("Precision:", round(float(precision_score(y, yhat, zero_division=0)), 3))
        print("Balanced Accuracy:", round(float(balanced_accuracy_score(y, yhat)), 3))
        print("Accuracy:", round(float(accuracy_score(y, yhat)), 3))
        return

    # If unlabeled (score mode), print prediction distribution
    df_ok = df_for_scoring.dropna(subset=FEATURES_ALL).copy()
    if df_ok.empty:
        print("No usable rows to score (features missing).")
        return

    X = df_ok[FEATURES_ALL]
    p = pipe.predict_proba(X)[:, 1]
    q = np.quantile(p, [0.01, 0.05, 0.50, 0.95, 0.99])

    print("Rows scored:", len(df_ok))
    print(
        f"risk_prob percentiles: "
        f"p01={q[0]:.3f} p05={q[1]:.3f} p50={q[2]:.3f} p95={q[3]:.3f} p99={q[4]:.3f}"
    )
    print("Share flagged (>= ALERT_THRESHOLD):", round(float((p >= ALERT_THRESHOLD).mean()), 4))


# =========================
# PIPELINE
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUM_FEATURES),
        ("cat", categorical_transformer, CAT_FEATURES),
    ],
    remainder="drop",
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    )),
])


# =========================
# TRAIN OR SCORE
# =========================
if MODE == "train":
    daily = build_daily_features(df_final)
    daily = add_soc_threshold_label(daily)

    df_model = daily.dropna(subset=NUM_FEATURES + ["y_risk"]).copy()
    train, test = session_split(df_model)

    X_train, y_train = train[FEATURES_ALL], train["y_risk"].astype(int)
    X_test,  y_test  = test[FEATURES_ALL],  test["y_risk"].astype(int)

    pipe.fit(X_train, y_train)

    test_scored = test.copy()
    test_scored["y_risk"] = y_test.values

    print_model_score_or_summary(pipe, test_scored, title="MODEL SCORE (TEST)")
    print_variable_importance(pipe, top_n=30)

    joblib.dump(pipe, MODEL_PATH)
    print("\nSaved model:", MODEL_PATH)

elif MODE == "score":
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"Model not found at {MODEL_PATH}. Set MODE='train' once to create it."
        )

    pipe = joblib.load(MODEL_PATH)
    print("Loaded model:", MODEL_PATH)

    daily_new = build_daily_features(df_new_final)

    risk_today = (
        daily_new.sort_values("date")
        .groupby(["bloqs_id", "battery_name"], observed=True, sort=False)
        .tail(1)
        .copy()
    )

    risk_today["risk_prob"] = pipe.predict_proba(risk_today[FEATURES_ALL])[:, 1]
    risk_today["model_alert"] = (risk_today["risk_prob"] >= ALERT_THRESHOLD).astype(int)

    risk_today["risk_score_voltage"] = np.select(
        [
            risk_today["volt_mean"] < 12.7,
            risk_today["volt_mean"].between(12.7, 12.9, inclusive="both"),
        ],
        ["Warning", "Early Warning"],
        default="OK",
    )

    print_model_score_or_summary(pipe, risk_today, title="MODEL SCORE (SCORE MODE SUMMARY)")
    print_variable_importance(pipe, top_n=30)

    export_cols = [
        "date", "bloqs_id", "battery_name", "country",
        "fw_version_type", "soc_mean", "volt_mean",
        "temp_mean", "rents_sum",
        "risk_score_voltage", "risk_prob", "model_alert",
        "address", "point_code",
    ]

    risk_today = risk_today.sort_values(["model_alert", "risk_prob"], ascending=[False, False])
    export_cols = [c for c in export_cols if c in risk_today.columns]

    risk_today[export_cols].to_csv(OUTPUT_CSV, index=False)
    print("\nExported:", OUTPUT_CSV)
    print("Rows exported:", len(risk_today))

    print("\nTop 10 scored rows:")
    print(risk_today[export_cols].head(10).to_string(index=False))

else:
    raise ValueError("MODE must be 'train' or 'score'")

Loaded model: /content/drive/MyDrive/battery_models/battery_risk_model.joblib

MODEL SCORE (SCORE MODE SUMMARY)
--------------------------------
Rows scored: 6913
risk_prob percentiles: p01=0.001 p05=0.011 p50=0.186 p95=0.909 p99=0.991
Share flagged (>= ALERT_THRESHOLD): 0.1008

[Importance] Top coefficients by |coef| (numerics are standardized):
num__volt_mean                -2.913287
num__soc_vol_7d                1.178429
num__volt_vol_7d              -0.871894
num__volt_slope_7d             0.679012
num__soc_slope_7d             -0.659140
num__volt_slope_3d             0.608095
cat__fw_version_type_recent   -0.601926
cat__fw_version_type_old      -0.408260
num__soc_slope_3d             -0.094107
num__rents_sum                -0.059814
num__temp_mean                 0.004761

Exported: battery_risk_20260421.csv
Rows exported: 7081

Top 10 scored rows:
      date                 bloqs_id battery_name country fw_version_type  soc_mean  volt_mean  temp_mean  rents_sum risk_score_voltag